# UMTAM Multi-Task Model Merging Experiment

**Objective:** Validate UMTAM's unified training-merging framework (Algorithm 2)

**Approach:**
1. Train 4 task-specific BERT models on GLUE tasks (RTE, MRPC, QNLI, CoLA)
2. Collect curvature statistics during training for each
3. Merge models using multiple methods:
   - **UMTAM** (Curvature-aware with sign election)
   - Linear Averaging (Task Arithmetic)
   - TIES-Merging
   - DARE
4. Evaluate merged models on ALL tasks simultaneously

**Expected Runtime:** ~2-3 hours on L4 GPU

**Key Insight:** UMTAM reuses curvature information from training for principled merging,
while baselines must infer importance from final parameters alone.

---
## 📦 Setup

In [ ]:
# Install packages
!pip install -q transformers datasets torch evaluate matplotlib seaborn scipy
print("✓ Packages installed")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from datasets import load_dataset
import evaluate
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import json
import copy
import warnings
import gc
from collections import defaultdict
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✓ Device: {device}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## ⚙️ Configuration

In [ ]:
CONFIG = {
    # Model
    'model_name': 'bert-base-uncased',
    
    # Tasks to merge (subset for faster experimentation)
    'tasks': {
        'rte': {
            'num_labels': 2,
            'metric': 'accuracy',
            'glue_name': 'rte',
            'text_fields': ('sentence1', 'sentence2'),
        },
        'mrpc': {
            'num_labels': 2,
            'metric': 'f1',
            'glue_name': 'mrpc',
            'text_fields': ('sentence1', 'sentence2'),
        },
        'qnli': {
            'num_labels': 2,
            'metric': 'accuracy',
            'glue_name': 'qnli',
            'text_fields': ('question', 'sentence'),
        },
        'cola': {
            'num_labels': 2,
            'metric': 'matthews_correlation',
            'glue_name': 'cola',
            'text_fields': ('sentence', None),
        },
    },
    
    # Training
    'max_length': 128,
    'batch_size': 32,
    'num_epochs': 3,
    'lr': 2e-5,
    
    # Merging
    'sparsity_k': 20,  # Keep top 20% by saliency (for UMTAM and TIES)
    'dare_p': 0.9,     # Drop 90% of parameters randomly (for DARE)
    
    # Other
    'seed': 42,
    'max_train_samples': 5000,  # Limit for faster training (set None for full)
    'max_eval_samples': 1000,   # Limit for faster eval
}

# Set seed
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

print("✓ Configuration:")
print(f"  Model: {CONFIG['model_name']}")
print(f"  Tasks: {list(CONFIG['tasks'].keys())}")
print(f"  Epochs per task: {CONFIG['num_epochs']}")
print(f"  Sparsity (k): {CONFIG['sparsity_k']}%")

---
## 📚 Load and Prepare Datasets

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

def load_and_prepare_dataset(task_name, task_config):
    """Load and tokenize a GLUE dataset"""
    print(f"\nLoading {task_name.upper()}...")
    dataset = load_dataset('glue', task_config['glue_name'])
    
    text1, text2 = task_config['text_fields']
    
    def tokenize_fn(examples):
        if text2 is None:
            result = tokenizer(
                examples[text1],
                padding='max_length',
                truncation=True,
                max_length=CONFIG['max_length']
            )
        else:
            result = tokenizer(
                examples[text1],
                examples[text2],
                padding='max_length',
                truncation=True,
                max_length=CONFIG['max_length']
            )
        result['labels'] = examples['label']
        return result
    
    tokenized = dataset.map(tokenize_fn, batched=True)
    
    # Subsample if configured
    train_data = tokenized['train'].shuffle(seed=CONFIG['seed'])
    if CONFIG['max_train_samples'] and len(train_data) > CONFIG['max_train_samples']:
        train_data = train_data.select(range(CONFIG['max_train_samples']))
    
    eval_data = tokenized['validation']
    if CONFIG['max_eval_samples'] and len(eval_data) > CONFIG['max_eval_samples']:
        eval_data = eval_data.select(range(CONFIG['max_eval_samples']))
    
    train_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    eval_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    
    print(f"  Train: {len(train_data)}, Eval: {len(eval_data)}")
    return train_data, eval_data

# Load all datasets
datasets = {}
for task_name, task_config in CONFIG['tasks'].items():
    train_data, eval_data = load_and_prepare_dataset(task_name, task_config)
    datasets[task_name] = {'train': train_data, 'eval': eval_data}

print(f"\n✓ Loaded {len(datasets)} datasets")

---
## 🏋️ Training with Curvature Collection

In [ ]:
def train_task_expert(task_name, task_config, train_dataset, base_state_dict):
    """
    Train a task-specific expert from shared initialization.
    Returns: trained state dict, task vector, saliency scores, curvature estimates
    """
    print(f"\n{'='*60}")
    print(f"Training {task_name.upper()} Expert")
    print(f"{'='*60}")
    
    # Create fresh model and load base weights
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG['model_name'],
        num_labels=task_config['num_labels']
    )
    
    # Load shared initialization for encoder (not classifier head)
    model_dict = model.state_dict()
    for name, param in base_state_dict.items():
        if name in model_dict and 'classifier' not in name:
            model_dict[name] = param.clone()
    model.load_state_dict(model_dict)
    
    model.to(device)
    model.train()
    
    optimizer = AdamW(model.parameters(), lr=CONFIG['lr'])
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
    
    # Store w0 (initial weights)
    w0_dict = {}
    for name, param in model.named_parameters():
        if len(param.shape) == 2 and 'classifier' not in name:  # Weight matrices in encoder
            w0_dict[name] = param.data.clone().cpu()
    
    # Initialize curvature tracking (R: row-wise, C: column-wise)
    curvature_dict = {}
    for name, param in model.named_parameters():
        if name in w0_dict:
            m, n = param.shape
            curvature_dict[name] = {
                'R': torch.zeros(m, device=device),
                'C': torch.zeros(n, device=device),
            }
    
    # Training loop
    for epoch in range(CONFIG['num_epochs']):
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']}")
        
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            
            # Accumulate curvature statistics (second moment of gradients)
            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in curvature_dict and param.grad is not None:
                        grad_sq = param.grad ** 2
                        curvature_dict[name]['R'] += grad_sq.sum(dim=1)
                        curvature_dict[name]['C'] += grad_sq.sum(dim=0)
            
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        print(f"  Epoch {epoch+1} avg loss: {total_loss / len(train_loader):.4f}")
    
    # Compute task vector: Δw = w_trained - w0
    task_vector = {}
    for name in w0_dict.keys():
        param = dict(model.named_parameters())[name]
        task_vector[name] = (param.data.cpu() - w0_dict[name]).clone()
    
    # Compute curvature-aware saliency scores
    saliency_dict = {}
    for name in w0_dict.keys():
        param = dict(model.named_parameters())[name]
        w0 = w0_dict[name].to(device)
        
        # Parameter change squared
        delta_sq = (param.data - w0) ** 2
        
        # Curvature: sqrt(R ⊗ C)
        R = curvature_dict[name]['R'].clamp(min=1e-8)
        C = curvature_dict[name]['C'].clamp(min=1e-8)
        curvature = torch.sqrt(R.unsqueeze(1) * C.unsqueeze(0))
        
        # Saliency = delta² × curvature
        saliency_dict[name] = (delta_sq * curvature).cpu()
    
    # Move curvature to CPU for merging
    curvature_cpu = {}
    for name in curvature_dict:
        curvature_cpu[name] = {
            'R': curvature_dict[name]['R'].cpu(),
            'C': curvature_dict[name]['C'].cpu(),
        }
    
    # Get trained state dict
    trained_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    # Cleanup
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    print(f"✓ {task_name.upper()} expert trained")
    
    return {
        'state_dict': trained_state,
        'task_vector': task_vector,
        'saliency': saliency_dict,
        'curvature': curvature_cpu,
        'w0': w0_dict,
    }

print("✓ Training function defined")

---
## 🔀 Merging Methods

In [ ]:
def merge_linear(base_state, task_experts, scaling=1.0):
    """
    Linear averaging (Task Arithmetic):
    w_merged = w0 + (1/K) * sum(task_vectors)
    """
    merged_state = copy.deepcopy(base_state)
    
    # Get layer names from first expert
    layer_names = list(task_experts[list(task_experts.keys())[0]]['task_vector'].keys())
    
    for name in layer_names:
        # Average task vectors
        avg_delta = torch.zeros_like(task_experts[list(task_experts.keys())[0]]['task_vector'][name])
        for task_name, expert in task_experts.items():
            avg_delta += expert['task_vector'][name]
        avg_delta /= len(task_experts)
        
        # Apply to base
        if name in merged_state:
            merged_state[name] = merged_state[name] + scaling * avg_delta
    
    return merged_state


def merge_ties(base_state, task_experts, k_percent=20):
    """
    TIES-Merging:
    1. Trim: Keep only top-k% magnitude parameters per task
    2. Elect Sign: Majority vote weighted by magnitude
    3. Disjoint Merge: Average only agreeing parameters
    """
    merged_state = copy.deepcopy(base_state)
    layer_names = list(task_experts[list(task_experts.keys())[0]]['task_vector'].keys())
    task_names = list(task_experts.keys())
    
    for name in layer_names:
        # Step 1: Trim - create masks for top-k% per task
        masks = {}
        trimmed_vectors = {}
        
        for task_name in task_names:
            tv = task_experts[task_name]['task_vector'][name]
            magnitude = tv.abs().flatten()
            k = int(len(magnitude) * k_percent / 100)
            threshold = torch.kthvalue(magnitude, len(magnitude) - k).values
            mask = (tv.abs() >= threshold).float()
            masks[task_name] = mask
            trimmed_vectors[task_name] = tv * mask
        
        # Step 2: Elect Sign
        pos_mass = torch.zeros_like(trimmed_vectors[task_names[0]])
        neg_mass = torch.zeros_like(trimmed_vectors[task_names[0]])
        
        for task_name in task_names:
            tv = trimmed_vectors[task_name]
            pos_mass += (tv > 0).float() * tv.abs()
            neg_mass += (tv < 0).float() * tv.abs()
        
        elected_sign = torch.sign(pos_mass - neg_mass)
        
        # Step 3: Disjoint Merge
        merged_delta = torch.zeros_like(trimmed_vectors[task_names[0]])
        count = torch.zeros_like(trimmed_vectors[task_names[0]])
        
        for task_name in task_names:
            tv = trimmed_vectors[task_name]
            # Only include if sign agrees with elected sign
            agree_mask = (torch.sign(tv) == elected_sign).float() * masks[task_name]
            merged_delta += tv * agree_mask
            count += agree_mask
        
        # Average where count > 0
        merged_delta = torch.where(count > 0, merged_delta / count.clamp(min=1), merged_delta)
        
        if name in merged_state:
            merged_state[name] = merged_state[name] + merged_delta
    
    return merged_state


def merge_dare(base_state, task_experts, drop_prob=0.9):
    """
    DARE (Drop And REscale):
    1. Randomly drop parameters with probability p
    2. Rescale remaining by 1/(1-p)
    3. Average rescaled task vectors
    """
    merged_state = copy.deepcopy(base_state)
    layer_names = list(task_experts[list(task_experts.keys())[0]]['task_vector'].keys())
    task_names = list(task_experts.keys())
    
    for name in layer_names:
        merged_delta = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        
        for task_name in task_names:
            tv = task_experts[task_name]['task_vector'][name]
            # Random mask
            mask = (torch.rand_like(tv) > drop_prob).float()
            # Rescale
            rescaled = tv * mask / (1 - drop_prob)
            merged_delta += rescaled
        
        merged_delta /= len(task_names)
        
        if name in merged_state:
            merged_state[name] = merged_state[name] + merged_delta
    
    return merged_state


def merge_umtam(base_state, task_experts, k_percent=20):
    """
    UMTAM Curvature-Aware Merging (Algorithm 2):
    1. Trim using curvature-aware saliency (not just magnitude)
    2. Elect sign weighted by saliency
    3. Curvature-weighted averaging
    """
    merged_state = copy.deepcopy(base_state)
    layer_names = list(task_experts[list(task_experts.keys())[0]]['task_vector'].keys())
    task_names = list(task_experts.keys())
    
    for name in layer_names:
        # Step 1: Create importance masks using curvature-aware saliency
        masks = {}
        
        for task_name in task_names:
            saliency = task_experts[task_name]['saliency'][name]
            flat_saliency = saliency.flatten()
            k = int(len(flat_saliency) * k_percent / 100)
            threshold = torch.kthvalue(flat_saliency, len(flat_saliency) - k).values
            masks[task_name] = (saliency >= threshold).float()
        
        # Step 2: Sign election weighted by saliency (not just magnitude)
        pos_mass = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        neg_mass = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        
        for task_name in task_names:
            tv = task_experts[task_name]['task_vector'][name]
            saliency = task_experts[task_name]['saliency'][name]
            mask = masks[task_name]
            
            # Weight by saliency, not magnitude
            pos_mass += (tv > 0).float() * saliency * mask
            neg_mass += (tv < 0).float() * saliency * mask
        
        elected_sign = torch.sign(pos_mass - neg_mass)
        
        # Step 3: Curvature-weighted merging
        # Compute preconditioner from curvature
        weighted_sum = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        weight_total = torch.zeros_like(task_experts[task_names[0]]['task_vector'][name])
        
        for task_name in task_names:
            tv = task_experts[task_name]['task_vector'][name]
            mask = masks[task_name]
            
            # Only include if sign agrees
            agree_mask = (torch.sign(tv) == elected_sign).float() * mask
            
            # Curvature weight: sqrt(R ⊗ C)
            R = task_experts[task_name]['curvature'][name]['R'].clamp(min=1e-8)
            C = task_experts[task_name]['curvature'][name]['C'].clamp(min=1e-8)
            curvature_weight = torch.sqrt(R.unsqueeze(1) * C.unsqueeze(0))
            curvature_weight = curvature_weight / (curvature_weight.max() + 1e-8)  # Normalize
            
            weighted_sum += tv * agree_mask * curvature_weight
            weight_total += agree_mask * curvature_weight
        
        # Weighted average
        merged_delta = torch.where(
            weight_total > 0,
            weighted_sum / weight_total.clamp(min=1e-8),
            torch.zeros_like(weighted_sum)
        )
        
        if name in merged_state:
            merged_state[name] = merged_state[name] + merged_delta
    
    return merged_state


def merge_dare_ties(base_state, task_experts, drop_prob=0.9, k_percent=20):
    """
    DARE + TIES: Apply DARE first, then TIES
    """
    merged_state = copy.deepcopy(base_state)
    layer_names = list(task_experts[list(task_experts.keys())[0]]['task_vector'].keys())
    task_names = list(task_experts.keys())
    
    for name in layer_names:
        # Step 1: DARE - drop and rescale
        dare_vectors = {}
        for task_name in task_names:
            tv = task_experts[task_name]['task_vector'][name]
            mask = (torch.rand_like(tv) > drop_prob).float()
            dare_vectors[task_name] = tv * mask / (1 - drop_prob)
        
        # Step 2: TIES on DARE vectors
        # Trim
        masks = {}
        for task_name in task_names:
            tv = dare_vectors[task_name]
            magnitude = tv.abs().flatten()
            k = int(len(magnitude) * k_percent / 100)
            if k > 0:
                threshold = torch.kthvalue(magnitude, len(magnitude) - k).values
                masks[task_name] = (tv.abs() >= threshold).float()
            else:
                masks[task_name] = torch.zeros_like(tv)
        
        # Elect sign
        pos_mass = torch.zeros_like(dare_vectors[task_names[0]])
        neg_mass = torch.zeros_like(dare_vectors[task_names[0]])
        
        for task_name in task_names:
            tv = dare_vectors[task_name] * masks[task_name]
            pos_mass += (tv > 0).float() * tv.abs()
            neg_mass += (tv < 0).float() * tv.abs()
        
        elected_sign = torch.sign(pos_mass - neg_mass)
        
        # Disjoint merge
        merged_delta = torch.zeros_like(dare_vectors[task_names[0]])
        count = torch.zeros_like(dare_vectors[task_names[0]])
        
        for task_name in task_names:
            tv = dare_vectors[task_name]
            agree_mask = (torch.sign(tv) == elected_sign).float() * masks[task_name]
            merged_delta += tv * agree_mask
            count += agree_mask
        
        merged_delta = torch.where(count > 0, merged_delta / count.clamp(min=1), merged_delta)
        
        if name in merged_state:
            merged_state[name] = merged_state[name] + merged_delta
    
    return merged_state


print("✓ Merging methods defined:")
print("  - Linear (Task Arithmetic)")
print("  - TIES-Merging")
print("  - DARE")
print("  - DARE+TIES")
print("  - UMTAM (Ours)")

---
## 📊 Evaluation Function

In [ ]:
def evaluate_on_task(model, task_name, task_config, eval_dataset):
    """Evaluate model on a specific task"""
    model.eval()
    model.to(device)
    
    eval_loader = DataLoader(eval_dataset, batch_size=CONFIG['batch_size'])
    metric = evaluate.load('glue', task_config['glue_name'])
    
    with torch.no_grad():
        for batch in eval_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = torch.argmax(outputs.logits, dim=-1)
            metric.add_batch(predictions=predictions, references=batch['labels'])
    
    result = metric.compute()
    return result[task_config['metric']]


def evaluate_merged_model(merged_state, base_state, task_name, task_config, eval_dataset):
    """
    Evaluate a merged model on a specific task.
    Creates a model with the merged encoder weights but task-specific classifier.
    """
    # Create model
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG['model_name'],
        num_labels=task_config['num_labels']
    )
    
    # Load merged state (encoder) - keep randomly initialized classifier
    model_dict = model.state_dict()
    for name, param in merged_state.items():
        if name in model_dict and 'classifier' not in name:
            model_dict[name] = param.clone()
    model.load_state_dict(model_dict)
    
    # We need to fine-tune the classifier briefly or use task-specific classifier
    # For fair comparison, we'll do a quick classifier-only fine-tune
    model.to(device)
    
    # Freeze encoder, train only classifier
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False
    
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
    train_loader = DataLoader(datasets[task_name]['train'], batch_size=CONFIG['batch_size'], shuffle=True)
    
    # Quick classifier training (1 epoch)
    model.train()
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        outputs.loss.backward()
        optimizer.step()
    
    # Evaluate
    score = evaluate_on_task(model, task_name, task_config, eval_dataset)
    
    del model
    torch.cuda.empty_cache()
    
    return score


def evaluate_all_tasks(merged_state, base_state, name="Model"):
    """Evaluate merged model on all tasks"""
    print(f"\nEvaluating {name}...")
    results = {}
    
    for task_name, task_config in CONFIG['tasks'].items():
        score = evaluate_merged_model(
            merged_state, base_state, 
            task_name, task_config, 
            datasets[task_name]['eval']
        )
        results[task_name] = score
        print(f"  {task_name.upper()}: {score:.4f}")
    
    # Compute average
    results['average'] = np.mean(list(results.values()))
    print(f"  AVERAGE: {results['average']:.4f}")
    
    return results

print("✓ Evaluation functions defined")

---
## 🚀 Run Experiment

In [ ]:
print("="*70)
print("UMTAM MULTI-TASK MERGING EXPERIMENT")
print("="*70)

# Step 1: Load base model and save initial state (w0)
print("\n[Step 1] Loading base model...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=2  # Dummy, will be replaced per task
)
base_state_dict = {k: v.cpu().clone() for k, v in base_model.state_dict().items()}
del base_model
torch.cuda.empty_cache()
print("✓ Base model state saved")

In [ ]:
# Step 2: Train task-specific experts
print("\n[Step 2] Training task-specific experts...")
task_experts = {}

for task_name, task_config in CONFIG['tasks'].items():
    expert = train_task_expert(
        task_name, 
        task_config, 
        datasets[task_name]['train'],
        base_state_dict
    )
    task_experts[task_name] = expert
    
    # Clear memory
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n✓ Trained {len(task_experts)} task experts")

In [ ]:
# Step 3: Evaluate individual experts (upper bound)
print("\n[Step 3] Evaluating individual experts (upper bound)...")
individual_results = {}

for task_name, task_config in CONFIG['tasks'].items():
    # Create model with expert weights
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG['model_name'],
        num_labels=task_config['num_labels']
    )
    model.load_state_dict(task_experts[task_name]['state_dict'])
    
    score = evaluate_on_task(
        model, task_name, task_config, 
        datasets[task_name]['eval']
    )
    individual_results[task_name] = score
    print(f"  {task_name.upper()} expert on {task_name.upper()}: {score:.4f}")
    
    del model
    torch.cuda.empty_cache()

print(f"\n✓ Individual expert average: {np.mean(list(individual_results.values())):.4f}")

In [ ]:
# Step 4: Apply merging methods
print("\n[Step 4] Applying merging methods...")

merging_methods = {
    'Linear': lambda: merge_linear(base_state_dict, task_experts),
    'TIES': lambda: merge_ties(base_state_dict, task_experts, k_percent=CONFIG['sparsity_k']),
    'DARE': lambda: merge_dare(base_state_dict, task_experts, drop_prob=CONFIG['dare_p']),
    'DARE+TIES': lambda: merge_dare_ties(base_state_dict, task_experts, 
                                          drop_prob=CONFIG['dare_p'], 
                                          k_percent=CONFIG['sparsity_k']),
    'UMTAM': lambda: merge_umtam(base_state_dict, task_experts, k_percent=CONFIG['sparsity_k']),
}

merged_states = {}
for method_name, merge_fn in merging_methods.items():
    print(f"\n  Merging with {method_name}...")
    merged_states[method_name] = merge_fn()
    print(f"  ✓ {method_name} complete")

print(f"\n✓ Created {len(merged_states)} merged models")

In [ ]:
# Step 5: Evaluate all merged models on all tasks
print("\n[Step 5] Evaluating merged models on all tasks...")
print("="*70)

all_results = {}

for method_name, merged_state in merged_states.items():
    results = evaluate_all_tasks(merged_state, base_state_dict, name=method_name)
    all_results[method_name] = results
    torch.cuda.empty_cache()
    gc.collect()

print("\n" + "="*70)
print("✓ EVALUATION COMPLETE")
print("="*70)

---
## 📊 Results Summary

In [ ]:
# Create results table
print("\n" + "="*90)
print("MULTI-TASK MERGING RESULTS")
print("="*90)

# Header
tasks = list(CONFIG['tasks'].keys())
header = f"{'Method':<15} | " + " | ".join([f"{t.upper():>8}" for t in tasks]) + " | " + f"{'AVG':>8}"
print(header)
print("-"*90)

# Individual experts row
expert_row = f"{'Experts (UB)':<15} | "
expert_row += " | ".join([f"{individual_results[t]:>8.4f}" for t in tasks])
expert_row += f" | {np.mean(list(individual_results.values())):>8.4f}"
print(expert_row)
print("-"*90)

# Merged models
for method_name, results in all_results.items():
    row = f"{method_name:<15} | "
    row += " | ".join([f"{results[t]:>8.4f}" for t in tasks])
    row += f" | {results['average']:>8.4f}"
    
    # Mark best merged method
    if method_name == max(all_results.keys(), key=lambda x: all_results[x]['average']):
        row += " ★"
    
    print(row)

print("="*90)
print("UB = Upper Bound (individual task-specific experts)")
print("★ = Best merging method")

In [ ]:
# Compute improvements over baselines
print("\n" + "="*70)
print("UMTAM vs BASELINES")
print("="*70)

umtam_avg = all_results['UMTAM']['average']

for method_name in ['Linear', 'TIES', 'DARE', 'DARE+TIES']:
    baseline_avg = all_results[method_name]['average']
    diff = umtam_avg - baseline_avg
    pct = (diff / baseline_avg) * 100 if baseline_avg > 0 else 0
    
    symbol = "✓" if diff > 0.001 else "≈" if abs(diff) < 0.001 else "✗"
    print(f"UMTAM vs {method_name:<12}: {diff:>+.4f} ({pct:>+.2f}%) {symbol}")

print("\n" + "-"*70)
print(f"UMTAM Average: {umtam_avg:.4f}")
print(f"Best Baseline: {max([all_results[m]['average'] for m in ['Linear', 'TIES', 'DARE', 'DARE+TIES']]):.4f}")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Per-task comparison
ax1 = axes[0]
x = np.arange(len(tasks))
width = 0.15

colors = {
    'Linear': '#A0A0A0',
    'TIES': '#5DA5DA',
    'DARE': '#FAA43A',
    'DARE+TIES': '#B276B2',
    'UMTAM': '#2E86AB',
}

for i, (method_name, results) in enumerate(all_results.items()):
    scores = [results[t] for t in tasks]
    ax1.bar(x + i*width, scores, width, label=method_name, color=colors[method_name])

ax1.set_xlabel('Task', fontsize=12, fontweight='bold')
ax1.set_ylabel('Score', fontsize=12, fontweight='bold')
ax1.set_title('Per-Task Performance by Merging Method', fontsize=13, fontweight='bold')
ax1.set_xticks(x + width * 2)
ax1.set_xticklabels([t.upper() for t in tasks])
ax1.legend(loc='lower right')
ax1.set_ylim(0, 1)

# Plot 2: Average comparison
ax2 = axes[1]
methods = list(all_results.keys())
averages = [all_results[m]['average'] for m in methods]

bars = ax2.bar(methods, averages, color=[colors[m] for m in methods])

# Highlight UMTAM
umtam_idx = methods.index('UMTAM')
bars[umtam_idx].set_edgecolor('gold')
bars[umtam_idx].set_linewidth(3)

ax2.set_xlabel('Method', fontsize=12, fontweight='bold')
ax2.set_ylabel('Average Score', fontsize=12, fontweight='bold')
ax2.set_title('Average Performance Across All Tasks', fontsize=13, fontweight='bold')
ax2.set_ylim(0, max(averages) * 1.15)

# Add value labels
for bar, avg in zip(bars, averages):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{avg:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('merging_results.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: merging_results.png")
plt.show()

In [ ]:
# Save results
results_to_save = {
    'config': CONFIG,
    'individual_experts': individual_results,
    'merged_results': all_results,
}

with open('merging_results.json', 'w') as f:
    json.dump(results_to_save, f, indent=2)
print("✓ Saved: merging_results.json")

print("\n" + "="*70)
print("EXPERIMENT COMPLETE!")
print("="*70)

---
## 🔬 Ablation: Varying Sparsity Levels

In [ ]:
# Test UMTAM at different sparsity levels
print("\n" + "="*70)
print("ABLATION: UMTAM Sparsity Sensitivity")
print("="*70)

sparsity_levels = [5, 10, 20, 40, 60, 80]
sparsity_results = {}

for k in sparsity_levels:
    print(f"\nTesting sparsity k={k}%...")
    merged = merge_umtam(base_state_dict, task_experts, k_percent=k)
    
    # Quick evaluation (just average)
    task_scores = []
    for task_name, task_config in CONFIG['tasks'].items():
        score = evaluate_merged_model(
            merged, base_state_dict,
            task_name, task_config,
            datasets[task_name]['eval']
        )
        task_scores.append(score)
    
    sparsity_results[k] = np.mean(task_scores)
    print(f"  Average: {sparsity_results[k]:.4f}")
    
    torch.cuda.empty_cache()

# Also test TIES at same levels for comparison
ties_sparsity_results = {}
for k in sparsity_levels:
    print(f"\nTesting TIES at k={k}%...")
    merged = merge_ties(base_state_dict, task_experts, k_percent=k)
    
    task_scores = []
    for task_name, task_config in CONFIG['tasks'].items():
        score = evaluate_merged_model(
            merged, base_state_dict,
            task_name, task_config,
            datasets[task_name]['eval']
        )
        task_scores.append(score)
    
    ties_sparsity_results[k] = np.mean(task_scores)
    print(f"  Average: {ties_sparsity_results[k]:.4f}")
    
    torch.cuda.empty_cache()

In [ ]:
# Plot sparsity ablation
plt.figure(figsize=(10, 6))

plt.plot(sparsity_levels, [sparsity_results[k] for k in sparsity_levels], 
         'o-', label='UMTAM', linewidth=2.5, markersize=10, color='#2E86AB')
plt.plot(sparsity_levels, [ties_sparsity_results[k] for k in sparsity_levels], 
         's--', label='TIES', linewidth=2.5, markersize=10, color='#5DA5DA')

# Mark improvements
for k in sparsity_levels:
    diff = sparsity_results[k] - ties_sparsity_results[k]
    if diff > 0.005:
        plt.annotate(f'+{diff:.3f}', 
                     (k, sparsity_results[k]),
                     textcoords="offset points",
                     xytext=(0, 10),
                     ha='center',
                     fontsize=9,
                     fontweight='bold',
                     color='green')

plt.xlabel('Sparsity Level k (%)', fontsize=12, fontweight='bold')
plt.ylabel('Average Score', fontsize=12, fontweight='bold')
plt.title('UMTAM vs TIES: Sensitivity to Sparsity Level', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sparsity_ablation.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: sparsity_ablation.png")
plt.show()

In [ ]:
# Final summary table
print("\n" + "="*60)
print("SPARSITY ABLATION SUMMARY")
print("="*60)
print(f"{'Sparsity k':>12} | {'UMTAM':>10} | {'TIES':>10} | {'Δ':>10}")
print("-"*60)

for k in sparsity_levels:
    umtam = sparsity_results[k]
    ties = ties_sparsity_results[k]
    diff = umtam - ties
    symbol = "✓" if diff > 0 else "✗"
    print(f"{k:>10}% | {umtam:>10.4f} | {ties:>10.4f} | {diff:>+10.4f} {symbol}")

print("="*60)
print(f"\nBest UMTAM sparsity: k={max(sparsity_results, key=sparsity_results.get)}%")
print(f"Best TIES sparsity: k={max(ties_sparsity_results, key=ties_sparsity_results.get)}%")